## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
from collections import OrderedDict
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import KernelDensity
from scipy.interpolate import UnivariateSpline
from scipy.signal import find_peaks
import statsmodels.api as sm 

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

sc.settings.verbosity = 0             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(facecolor='white', color_map='cividis_r', dpi_save=300, dpi=100)

## Load data

In [ ]:
adata = sc.read_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime.h5ad')
adata

In [ ]:
marker_genes_svz_dict = dict({
    "qNSC1": ["Aqp4", "Cxcl14"],
    "qNSC2": ["Cpe", "S100a6"],
    "aNSC1": ["Btg2", "Egr1"],
    "aNSC2": ["Egfr", "Ascl1", "Top2a"],
    "TAP": ["Mki67", "Ube2c"],
    "NB": ["Dcx", "Stmn2"],
    "Neurons": ["Rbfox3", "Calb2", "Nefl", "Th"],
    "Microglia": ["Itgam", "Ptprc"],
    "Oligodendrocytes": ["Cldn11"],
    "OPC": ["Olig1", "Olig2", "Lhfpl3", "Cntn1"],
    "Ependymal cells": ["Cald1", "Sod3"],
    "Olfactory Ensheathing cells": ["Apod"],
    "pericyte": ["Foxd1", "Colec12"],
    "Endothelial cells": ["Prom1", "Cldn5", "Pecam1"],
})

In [ ]:
marker_genes_svz_nolineage_dict = dict({
    "Neurons": ["Rbfox3", "Calb2", "Nefl", "Th"],
    "Microglia": ["Itgam", "Ptprc"],
    "Oligodendrocytes": ["Cldn11"],
    "OPC": ["Olig1", "Olig2", "Lhfpl3", "Cntn1"],
    "Ependymal cells": ["Cald1", "Sod3"],
    "Olfactory Ensheathing cells": ["Apod"],
    "pericyte": ["Foxd1", "Colec12"],
    "Endothelial cells": ["Prom1", "Cldn5", "Pecam1"],
})

In [ ]:
marker_genes_svz_lineage_dict = dict({
    "qNSC1": ["Aqp4", "Cxcl14"],
    "qNSC2": ["Cpe", "S100a6"],
    "aNSC1": ["Btg2", "Egr1"],
    "aNSC2": ["Egfr", "Ascl1", "Top2a"],
    "TAP": ["Mki67", "Ube2c"],
    "NB": ["Dcx", "Stmn2"],
})

In [ ]:
marker_genes_svz_list = [i for f in marker_genes_svz_dict.values() for i in f]
marker_genes_svz_nolineage_list = [i for f in marker_genes_svz_dict.values() for i in f]
marker_genes_svz_lineage_list = [i for f in marker_genes_svz_lineage_dict.values() for i in f]

In [ ]:
marker_genes_svz_dict_filtered = {key: [value for value in values if value in adata.var_names] for key, values in marker_genes_svz_dict.items()}
marker_genes_svz_nolineage_dict_filtered = {key: [value for value in values if value in adata.var_names] for key, values in marker_genes_svz_nolineage_dict.items()}

## Functions

In [ ]:
def normalize(x, scale=1):
    return (x - x.min(0)) / ((1/scale) * x.ptp(0))

def normalize_vector(data):
    return (data - np.min(data)) / (np.max(data) - np.min(data))

# Define cell types
## Leiden
- Leiden requires having run sc.tl.neighbors() first
- This dataset already contains a computed neighbors graph in `.uns[‘neighbors’]`
- This neighbors graph was computed via `sc.pp.neighbors(adata, n_neighbors=100, n_pcs=12, use_rep="X_scanorama")`
- And UMAP via `sc.tl.umap(adata, min_dist=0.3)`

In [ ]:
sc.tl.leiden(adata, resolution=1, random_state=0)

In [ ]:
sc.settings.set_figure_params(figsize=(10, 9))
sc.pl.umap(adata, color="leiden", s=20)

In [ ]:
sc.pl.umap(adata=adata, color=["dpt_pseudotime", "dbscan_clusters", "leiden", "phase"])

### Check leiden clusters

In [ ]:
sc.settings.set_figure_params(figsize=(7,5))
for celltype, markers in marker_genes_svz_dict_filtered.items():
    axs = sc.pl.umap(adata=adata, color=["leiden"] + markers, use_raw=False, show=False)
    for idx, ax in enumerate(axs):
        if idx == 0:
            continue 
        ax.set_ylabel("")
    plt.suptitle(celltype)

In [ ]:
sc.settings.set_figure_params(dpi=120, figsize=(12, 8))
sc.tl.dendrogram(adata, groupby="leiden", optimal_ordering=True)
sc.pl.tracksplot(adata, marker_genes_svz_dict, groupby='leiden', dendrogram=True)

In [ ]:
sc.pl.dotplot(adata, marker_genes_svz_dict, groupby="leiden", standard_scale='var', dendrogram=True)

In [ ]:
sc.pl.stacked_violin(adata, marker_genes_svz_dict, groupby="leiden", dendrogram=True)

## Attribute celltypes outside the lineage

In [ ]:
sc.settings.set_figure_params(figsize=(10, 10))
sc.pl.umap(adata, color=["leiden", "dbscan_clusters"], s=20)

In [ ]:
sc.settings.set_figure_params(dpi=120, figsize=(10, 10))
sc.pl.dotplot(adata, marker_genes_svz_nolineage_dict, groupby="leiden", standard_scale='var', dendrogram=True)

In [ ]:
sc.settings.set_figure_params(dpi=120, figsize=(10, 10))
sc.pl.dotplot(adata, marker_genes_svz_nolineage_dict, groupby="dbscan_clusters", standard_scale='var', dendrogram=True)

In [ ]:
marker_genes_svz_dict

In [ ]:
leiden = adata.obs.leiden
adata.obs["celltype"] = np.where(
    leiden == "12", "Microglia",
    np.where(
        leiden == "16", "Endothelial cells",
        np.where(
            leiden == "8", "Olfactory Ensheathing cells",
            np.where(
                leiden == "17", "OPC",
                np.where(
                    leiden == "13", "Neurons",
                    np.where(
                        leiden == "14", "Oligodendrocytes",
                        np.where(
                            leiden == "15", "Ependymal cells",
                            "Lineage"
                        )
                    )
                )
            )
        )
    )
)

In [ ]:
adata.obs["celltype"] = adata.obs.celltype.astype("category")

In [ ]:
sc.settings.set_figure_params(dpi=120, figsize=(10, 10))
sc.tl.dendrogram(adata, groupby="celltype", optimal_ordering=True)
sc.pl.dotplot(adata, marker_genes_svz_nolineage_dict, groupby="celltype", standard_scale='var', dendrogram=True)

In [ ]:
adata_nolineage = adata[adata.obs.celltype != "Lineage"].copy()
sc.tl.dendrogram(adata_nolineage, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_nolineage, marker_genes_svz_nolineage_dict, groupby="celltype", dendrogram=True)

In [ ]:
sc.settings.set_figure_params(figsize=(10, 10))
sc.pl.umap(adata, color=["celltype"], s=30, legend_loc="on data")

- Celltype attribution outside of the lineage seems overall sensible and correct.
- May take another look at the endothelial cell cluster that can contain also pericytes.

## PAGA

In [ ]:
adata_lineage = adata[adata.obs.celltype == "Lineage"].copy()
sc.pl.umap(adata_lineage, color="leiden", legend_loc="on data")

In [ ]:
sc.tl.paga(adata=adata_lineage, groups="leiden")

In [ ]:
sc.pl.paga(adata_lineage, color=["leiden", "celltype"], node_size_scale=3, threshold=0.1, random_state=1)

In [ ]:
sc.settings.set_figure_params(figsize=(10, 10))

sc.pl.paga_path(
    adata_lineage, 
    nodes=["11", "0", "6", "5", "7", "3", "2", "1"], 
    keys=marker_genes_svz_list[:13], 
    color_map='Greys', 
    color_maps_annotations={'dpt_pseudotime': 'viridis'}, 
    use_raw=False)

## Attribute lineage celltypes

### Mostly markers from Kalamakis 2019

In [ ]:
adata_lineage_raw = adata_lineage.raw.to_adata()
sc.pp.normalize_per_cell(adata_lineage_raw)
sc.pp.log1p(adata_lineage_raw)
lineage_markers_df = adata_lineage_raw.to_df()
lineage_markers_df = lineage_markers_df[marker_genes_svz_lineage_list]
lineage_markers_df["ptime"] = adata_lineage.obs.dpt_pseudotime
lineage_markers_df["leiden"] = adata_lineage.obs.leiden
lineage_markers_df["cluster"] = adata_lineage.obs.celltype
lineage_markers_df = lineage_markers_df.sort_values("ptime")
lineage_markers_df = lineage_markers_df.dropna(subset="ptime")

for gene in lineage_markers_df.columns[:-3]:
    value = lineage_markers_df[gene]
    value = value.rolling(window=1000, center=False).mean()
    lineage_markers_df[gene] = (value-value.min())/(value.max() - value.min())

lineage_markers_df

In [ ]:
marker_genes_svz_lineage_dict_inverted = {v: k for k, values in marker_genes_svz_lineage_dict.items() for v in values}
marker_genes_svz_lineage_dict_inverted

In [ ]:
sc.pl.umap(adata_lineage_raw, color=["leiden", "Btg2", "Egr1", "Top2a", "Egfr", "Notum", "Sfrp1", "S100a6", "Cpe", "Ube2c"], use_raw=False)

In [ ]:
lineage_markers_df_melt = pd.melt(lineage_markers_df, id_vars=["ptime", "leiden", "cluster"])
lineage_markers_df_melt = lineage_markers_df_melt.rename(columns={"variable": "Gene"})
lineage_markers_df_melt["Marker for"] = lineage_markers_df_melt["Gene"].map(marker_genes_svz_lineage_dict_inverted)
lineage_markers_df_melt

In [ ]:
# number of cells over pseudotime:
fig, ax = plt.subplots(1,1, figsize=(12, 8))
sns.lineplot(data=lineage_markers_df_melt, x="ptime", y="value", size="Gene", hue="Marker for", ax=ax, legend="brief", sizes=(.99, 1.01))
ax = sns.rugplot(data=lineage_markers_df, x="ptime", alpha=.05, c="k", hue="leiden", legend=False)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:7], labels[1:7])
sns.move_legend(ax, "center right", title="Marker", alignment="center")

ax.grid(False)
ax.set_ylabel("Expression")
ax.set_xlim(0, 1)
fig.set_tight_layout(True)

In [ ]:
""" Other methods for smoothing
# Binning
lineage_markers_df["ptime"] = pd.cut(
    lineage_markers_df.ptime, bins=100, labels=range(0, 100))
lineage_markers_df = lineage_markers_df.groupby("ptime").mean()

# Lowess
lowess = sm.nonparametric.lowess
for gene in lineage_markers_df.columns[:-3]:
    z = lowess(lineage_markers_df[gene], lineage_markers_df.ptime, frac=1 / 4, it=3)
    lineage_markers_df[gene] = [i[1] for i in z]

# Splines
for gene in lineage_markers_df.columns[:-3]:
    spl = UnivariateSpline(lineage_markers_df.ptime, lineage_markers_df[gene], k=3, s=5000)
   lineage_markers_df[gene] = spl(lineage_markers_df.ptime)
"""

### Most sensible markers

In [ ]:
marker_genes_svz_lineage_sensible_dict = dict({
    "qNSC1": ["Aqp4", "Cxcl14"],
    "qNSC2": ["Cpe", "S100a6", "Sfrp1", "Notum"],
    "aNSC1": ["Btg2", ],
    "aNSC2": ["Ascl1", "Egfr", "Top2a"],
    "TAP": ["Mki67", "Ube2c"],
    "NB": ["Dcx", "Stmn2"],
})

marker_genes_svz_lineage_sensible_list = [i for f in marker_genes_svz_lineage_sensible_dict.values() for i in f]

In [ ]:
adata_lineage_raw = adata_lineage.raw.to_adata()
sc.pp.normalize_per_cell(adata_lineage_raw)
sc.pp.log1p(adata_lineage_raw)
lineage_markers_df = adata_lineage_raw.to_df()
lineage_markers_df = lineage_markers_df[marker_genes_svz_lineage_sensible_list]
lineage_markers_df["ptime"] = adata_lineage.obs.dpt_pseudotime
lineage_markers_df["leiden"] = adata_lineage.obs.leiden
lineage_markers_df["cluster"] = adata_lineage.obs.celltype
lineage_markers_df = lineage_markers_df.sort_values("ptime")
lineage_markers_df = lineage_markers_df.dropna(subset="ptime")

for gene in lineage_markers_df.columns[:-3]:
    value = lineage_markers_df[gene]
    value = value.rolling(window=1000, center=True).mean()
    lineage_markers_df[gene] = (value-value.min())/(value.max() - value.min())

lineage_markers_df

In [ ]:
marker_genes_svz_lineage_dict_inverted = {v: k for k, values in marker_genes_svz_lineage_sensible_dict.items() for v in values}
marker_genes_svz_lineage_dict_inverted

In [ ]:
lineage_markers_df_melt = pd.melt(lineage_markers_df, id_vars=["ptime", "leiden", "cluster"])
lineage_markers_df_melt = lineage_markers_df_melt.rename(columns={"variable": "Gene"})
lineage_markers_df_melt["Marker for"] = lineage_markers_df_melt["Gene"].map(marker_genes_svz_lineage_dict_inverted)
lineage_markers_df_melt

In [ ]:
# number of cells over pseudotime:
fig, ax = plt.subplots(1,1, figsize=(12, 8))
sns.lineplot(data=lineage_markers_df_melt, x="ptime", y="value", size="Gene", hue="Marker for", ax=ax, legend="brief", sizes=(.99, 1.01))
ax = sns.rugplot(data=lineage_markers_df, x="ptime", alpha=.05, c="k", hue="leiden", legend=False)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:7], labels[1:7])
sns.move_legend(ax, "center right", title="Marker", alignment="center")

# ax.axvline(0.1, color="k")
# ax.axvline(0.46, color="k")
# ax.axvline(0.63, color="k")
# ax.axvline(0.8, color="k")
# ax.axvline(0.9, color="k")

ax.axvline(0.077, color="k")
ax.axvline(0.25, color="k")
ax.axvline(0.48, color="k")
ax.axvline(0.78, color="k")
ax.axvline(0.88, color="k")

ax.grid(False)
ax.set_ylabel("Expression")
ax.set_xlim(0, 1)
fig.set_tight_layout(True)

In [ ]:
colors_leiden = {l: c for l, c in zip(lineage_markers_df.leiden.unique(), sns.color_palette(n_colors=12).as_hex())}
colors_leiden

In [ ]:
celltype_divisions = [0.068, 0.2, 0.565, 0.78, 0.867]

In [ ]:
f = plt.figure(figsize=(12, 7 + (7*0.15)))
grid = f.add_gridspec(ncols=1, nrows=2, 
         height_ratios=[0.15, 1], hspace=0.03)

ax1 = f.add_subplot(grid[0,0])
ax2 = f.add_subplot(grid[1,0])

x = np.linspace(0, 1, lineage_markers_df.shape[0])
kde = KernelDensity(kernel='gaussian', bandwidth=0.02).fit(lineage_markers_df['ptime'].dropna().values.reshape(-1, 1))
kde = np.exp(kde.score_samples(x.reshape(-1, 1)))
Z = normalize(kde, scale=1)

celltype_boundaries = OrderedDict()
for celltype in lineage_markers_df.sort_values("ptime").leiden.unique():
    df_celltype = lineage_markers_df.copy()
    df_celltype = df_celltype.sort_values("ptime")
    df_celltype["indexes"] = range(1, df_celltype.shape[0] + 1)
    df_celltype = df_celltype.query("leiden == @celltype")
    celltype_boundaries[celltype] = df_celltype.query("ptime <= @df_celltype.ptime.quantile(q=0.95)").indexes[-1]
last_item = list(celltype_boundaries.keys())[-1]
celltype_boundaries[last_item] = df_celltype.indexes[-1]

ax1.plot(x, Z, color='k', linewidth=1)

current_idx = 0
for cluster, idx in celltype_boundaries.items():
    ax1.fill_between(x[current_idx:idx+1], 0, Z[current_idx:idx+1], alpha=0.4,
                     color=colors_leiden[cluster])
    current_idx = idx

minima = find_peaks(Z * -1)[0]

for i in x[minima]:
    ax1.axvline(i, c='lightgray', ls='--', lw=2)
    
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_visible(False)
ax1.set_xlim(-0.01, 1.01)
ax1.set_yticks([])
ax1.set_xticks([])

########################################################################

sns.lineplot(data=lineage_markers_df_melt, x="ptime", y="value", size="Gene", hue="Marker for", ax=ax2, legend="brief", sizes=(.99, 1.01))
ax2 = sns.rugplot(data=lineage_markers_df, x="ptime", alpha=.05, c="k", hue="leiden", legend=False)

handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles[1:7], labels[1:7])
sns.move_legend(ax2, "center right", title="Marker", alignment="center")

# ax2.axvline(0.1, color="k")
# ax2.axvline(0.46, color="k")
# ax2.axvline(0.63, color="k")
# ax2.axvline(0.8, color="k")
# ax2.axvline(0.9, color="k")

for division in celltype_divisions: ax2.axvline(division, color="k")

ax2.grid(False)
ax2.set_ylabel("Expression")
ax2.set_xlim(0, 1)

########################################################################

plt.tight_layout()
plt.show()

In [ ]:
ptime = adata_lineage_raw.obs.dpt_pseudotime
adata_lineage_raw.obs["celltype"] = np.where(
    ptime < celltype_divisions[0], "qNSC1",
    np.where(ptime < celltype_divisions[1], "qNSC2",
             np.where(ptime < celltype_divisions[2], "aNSC1",
                      np.where(ptime < celltype_divisions[3], "aNSC2",
                               np.where(ptime < celltype_divisions[4], "TAP",
                                        "NB"))))
)
adata_lineage_raw.obs

In [ ]:
sc.settings.set_figure_params(figsize=(5, 5), dpi=100)
sc.pl.umap(adata_lineage_raw, color=["celltype", "leiden"] + marker_genes_svz_lineage_sensible_list, s=20)

In [ ]:
colors_celltypes = {l: c for l, c in zip(adata_lineage_raw.obs.celltype.unique(), adata_lineage_raw.uns["celltype_colors"])}
colors_celltypes

In [ ]:
sc.settings.set_figure_params(dpi=120)
f = plt.figure(figsize=(10, 8 + (7*0.15)))
grid = f.add_gridspec(ncols=1, nrows=2, 
         height_ratios=[0.15, 1], hspace=0.03)

ax1 = f.add_subplot(grid[0,0])
ax2 = f.add_subplot(grid[1,0])

df = adata_lineage_raw.obs.copy()

x = np.linspace(0, 1, df.shape[0])
kde = KernelDensity(kernel='gaussian', bandwidth=0.02).fit(df['dpt_pseudotime'].dropna().values.reshape(-1, 1))
kde = np.exp(kde.score_samples(x.reshape(-1, 1)))
Z = normalize(kde, scale=1)

celltype_boundaries = OrderedDict()
for celltype in df.sort_values("dpt_pseudotime").celltype.unique():
    df_celltype = df.copy()
    df_celltype = df_celltype.sort_values("dpt_pseudotime")
    df_celltype["indexes"] = range(1, df_celltype.shape[0] + 1)
    df_celltype = df_celltype.query("celltype == @celltype")
    celltype_boundaries[celltype] = df_celltype.query("dpt_pseudotime <= @df_celltype.dpt_pseudotime.quantile(q=0.95)").indexes[-1]
last_item = list(celltype_boundaries.keys())[-1]
celltype_boundaries[last_item] = df_celltype.indexes[-1]

ax1.plot(x, Z, color='k', linewidth=1)

current_idx = 0
for cluster, idx in celltype_boundaries.items():
    ax1.fill_between(x[current_idx:idx+1], 0, Z[current_idx:idx+1], alpha=0.4,
                     color=colors_celltypes[cluster])
    current_idx = idx

minima = find_peaks(Z * -1)[0]

for i in x[minima]:
    ax1.axvline(i, c='lightgray', ls='--', lw=2)
    
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_visible(False)
ax1.set_xlim(-0.01, 1.01)
ax1.set_yticks([])
ax1.set_xticks([])

########################################################################

sc.pl.umap(adata_lineage_raw, color=["celltype"], s=30, ax=ax2, title="")

In [ ]:
celltype_categories_reordered = [marker for marker in marker_genes_svz_lineage_sensible_dict.keys() if marker in adata_lineage_raw.obs.celltype.cat.categories]
adata_lineage_raw.obs.celltype.cat.reorder_categories(celltype_categories_reordered, ordered=True, inplace=True)
adata_lineage_raw.obs.celltype

In [ ]:
adata_lineage_sorted = adata_lineage_raw[adata_lineage_raw.obs.sort_values(["celltype", "dpt_pseudotime"], ascending=True).index].copy()
adata_lineage_sorted.obs

In [ ]:
sc.settings.set_figure_params(figsize=(15, 12), dpi=120)
sc.pl.heatmap(adata_lineage_raw, marker_genes_svz_lineage_sensible_dict, groupby='celltype', swap_axes=True)

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white')
sc.tl.dendrogram(adata_lineage_sorted, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_lineage_sorted, marker_genes_svz_lineage_sensible_dict, groupby='celltype', dendrogram=False)

In [ ]:
sc.settings.set_figure_params(figsize=(5, 5), dpi=100)
sc.pl.umap(adata_lineage_raw, color=["celltype", "leiden", "batch"], s=20)

In [ ]:
sc.settings.set_figure_params(figsize=(15, 12), dpi=100)
fig, axs = plt.subplots(nrows=2, ncols=3)
for ax, batch in zip(axs.flatten(), adata_lineage_raw.obs.batch.unique()):
    adata_batch = adata_lineage_raw[adata_lineage_raw.obs.batch == batch]
    sc.pl.umap(adata_batch, color="celltype", s=20, ax=ax, show=False)

### Re-assign the qNSC1/qNSC2 split according to leiden clusters

In [ ]:
adata_lineage_backup = adata_lineage_raw.copy()

adata_lineage_raw.obs["leiden_mix"] = np.where(
    adata_lineage_raw.obs.leiden == "11", "Astrocytes",
    np.where(adata_lineage_raw.obs.leiden == "0", "qNSC1",
             np.where(
                 adata_lineage_raw.obs.leiden == "6", "qNSC2",
                 np.where(
                     adata_lineage_raw.obs.leiden == "5", "aNSC1",
                     adata_lineage_raw.obs.leiden
                 )
             )
             )
)

celltypes = adata_lineage_raw.obs.celltype
leiden_mix = adata_lineage_raw.obs.leiden_mix
adata_lineage_raw.obs["celltype"] = pd.Categorical(
    np.where(celltypes == "qNSC1", leiden_mix,
             np.where((celltypes == "qNSC2") & (leiden_mix.isin(["qNSC2", "Astrocytes", "qNSC1"])), leiden_mix,
                      celltypes
                      )
             )
)
adata_lineage_raw.obs = adata_lineage_raw.obs.drop("leiden_mix", axis=1)
adata_lineage_raw.obs

In [ ]:
sc.settings.set_figure_params(figsize=(4, 4), dpi=100)
sc.pl.umap(adata_lineage_raw, color=["celltype", "leiden"], ncols=1)

In [ ]:
celltype_categories_reordered = [marker for marker in ["Astrocytes"] + list(marker_genes_svz_lineage_sensible_dict.keys()) if marker in adata_lineage_raw.obs.celltype.cat.categories]
adata_lineage_raw.obs.celltype.cat.reorder_categories(celltype_categories_reordered, ordered=True, inplace=True)
adata_lineage_sorted = adata_lineage_raw[adata_lineage_raw.obs.sort_values(["celltype", "dpt_pseudotime"], ascending=True).index].copy()
adata_lineage_sorted.obs

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white')
sc.tl.dendrogram(adata_lineage_sorted, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_lineage_sorted, marker_genes_svz_lineage_sensible_dict, groupby='celltype', dendrogram=False)

# Assign the defined celltypes to the SVZ data set

In [ ]:
adata_backup = adata.copy()

In [ ]:
adata.obs = adata.obs.join(adata_lineage_raw.obs.celltype, lsuffix="_old")
adata.obs["celltype_new"] = pd.Categorical(np.where(adata.obs.celltype_old=="Lineage", adata.obs.celltype, adata.obs.celltype_old))
adata.obs

In [ ]:
celltype_categories_reordered = [marker for marker in ["Astrocytes"] + list(marker_genes_svz_dict.keys()) if marker in adata.obs.celltype_new.cat.categories]
adata.obs.celltype = adata.obs.celltype_new.cat.reorder_categories(celltype_categories_reordered, ordered=False)
adata.obs = adata.obs.drop(["celltype_new", "celltype_old"], axis=1)
adata.obs

In [ ]:
adata.obs.celltype.unique()

In [ ]:
adata

In [ ]:
adata.write_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes.h5ad')